# 00_core.ipynb

In [ ]:
%%capture
!pip install -q --progress-bar off -r ../requirements.txt

In [ ]:
from dataclasses import dataclass
from itertools import combinations
from math import ceil
from pathlib import Path
from time import perf_counter

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# from fim import apriori, eclat, fpgrowth

## Cấu hình

In [ ]:
@dataclass
class ProjectConfig:
    dataset: str = "psparks/instacart-market-basket-analysis"
    min_basket_size: int = 2
    min_support: float = 0.003
    min_confidence: float = 0.20
    min_lift: float = 1.00
    top_n: int = 15


CFG = ProjectConfig()

## Đường dẫn

In [ ]:
class ProjectPaths:
    def __init__(self):
        cwd = Path.cwd()
        self.root = cwd.parent if cwd.name == "notebooks" else cwd
        self.outputs = self.root / "outputs"
        self.data = self.outputs / "data"
        self.tables = self.outputs / "tables"
        self.figures = self.outputs / "figures"
        self.app = self.root / "app"
        self.make_dirs()

    def make_dirs(self):
        for path in [self.data, self.tables, self.figures, self.app]:
            path.mkdir(parents=True, exist_ok=True)

    def data_file(self, name):
        return self.data / name

    def table_file(self, name):
        return self.tables / name

    def figure_file(self, name):
        return self.figures / name


PATHS = ProjectPaths()

## Lưu và đọc bảng

In [ ]:
def save_csv(data, path):
    data.to_csv(path, index=False, encoding="utf-8-sig")


def load_csv(path):
    return pd.read_csv(path)

## Đọc dữ liệu

In [ ]:
class DataLoader:
    files = {
        "orders": "orders.csv",
        "prior": "order_products__prior.csv",
        "train": "order_products__train.csv",
        "products": "products.csv",
        "aisles": "aisles.csv",
        "departments": "departments.csv",
    }

    def __init__(self, cfg=CFG):
        self.cfg = cfg

    def dataset_path(self):
        return Path(kagglehub.dataset_download(self.cfg.dataset))

    def locate_files(self):
        base = self.dataset_path()
        found = {}

        for key, filename in self.files.items():
            matches = list(base.rglob(filename))
            if matches:
                found[key] = matches[0]

        missing = [key for key in self.files if key not in found]
        if missing:
            raise FileNotFoundError(f"Không tìm thấy file dữ liệu: {missing}")

        return found

    def load_raw(self):
        files = self.locate_files()
        return {key: pd.read_csv(path) for key, path in files.items()}

## Chuẩn bị dữ liệu

In [ ]:
class DataPrep:
    def __init__(self, cfg=CFG):
        self.cfg = cfg

    def clean_product_name(self, value):
        value = str(value)
        value = value.replace('\\"', '"')
        value = value.replace("\xa0", " ")
        value = " ".join(value.split())
        return value.strip()

    def clean(self, raw):
        rows = []
        cleaned = {}

        for name, data in raw.items():
            before_rows = len(data)
            before_missing = int(data.isna().sum().sum())
            before_duplicates = int(data.duplicated().sum())

            data = data.drop_duplicates().copy()

            if name == "products" and "product_name" in data:
                data["product_name"] = data["product_name"].apply(
                    self.clean_product_name
                )

            if name == "orders" and "days_since_prior_order" in data:
                data["days_since_prior_order"] = data[
                    "days_since_prior_order"
                ].fillna(0)

            after_rows = len(data)
            after_missing = int(data.isna().sum().sum())
            after_duplicates = int(data.duplicated().sum())
            cleaned[name] = data

            rows.append([
                name,
                before_rows,
                after_rows,
                before_rows - after_rows,
                before_missing,
                after_missing,
                before_duplicates,
                after_duplicates,
            ])

        columns = [
            "Tập dữ liệu",
            "Số dòng ban đầu",
            "Số dòng sau xử lý",
            "Số dòng bị xóa",
            "Giá trị thiếu ban đầu",
            "Giá trị thiếu sau xử lý",
            "Dòng trùng ban đầu",
            "Dòng trùng sau xử lý",
        ]
        return cleaned, pd.DataFrame(rows, columns=columns)

    def product_catalog(self, data):
        catalog = data["products"].merge(data["aisles"], on="aisle_id")
        catalog = catalog.merge(data["departments"], on="department_id")
        return catalog.rename(columns={
            "product_name": "Sản phẩm",
            "aisle": "Nhóm hàng chi tiết",
            "department": "Nhóm hàng lớn",
        })

    def prior_items(self, data):
        columns = ["order_id", "product_id", "reordered"]
        return data["prior"][columns].copy()

    def train_items(self, data):
        columns = ["order_id", "product_id", "reordered"]
        return data["train"][columns].copy()

    def order_time(self, data):
        columns = [
            "order_id",
            "user_id",
            "eval_set",
            "order_number",
            "order_dow",
            "order_hour_of_day",
            "days_since_prior_order",
        ]
        orders = data["orders"][columns].copy()
        return orders[orders["eval_set"].eq("prior")].copy()

    def data_size(self, prior_items, train_items):
        basket_size = prior_items.groupby("order_id")["product_id"].nunique()
        valid_baskets = basket_size[basket_size >= self.cfg.min_basket_size]
        valid_items = prior_items[
            prior_items["order_id"].isin(valid_baskets.index)
        ]
        labels = [
            "Đơn hàng prior ban đầu",
            "Giỏ prior đủ điều kiện",
            "Giỏ prior bị loại vì chỉ có 1 sản phẩm",
            "Tổng số sản phẩm trong các giỏ prior đủ điều kiện",
            "Sản phẩm khác nhau xuất hiện trong prior",
            "Đơn hàng train dùng để kiểm chứng",
            "Tổng số sản phẩm trong train",
            "Sản phẩm khác nhau xuất hiện trong train",
        ]
        values = [
            int(len(basket_size)),
            int(len(valid_baskets)),
            int(len(basket_size) - len(valid_baskets)),
            int(len(valid_items)),
            int(valid_items["product_id"].nunique()),
            int(train_items["order_id"].nunique()),
            int(len(train_items)),
            int(train_items["product_id"].nunique()),
        ]
        return pd.DataFrame({
            "Chỉ tiêu": labels,
            "Số lượng": values,
        })

## Phân tích hành vi

In [ ]:
class BehaviorAnalyzer:
    day_names = {
        0: "Chủ nhật",
        1: "Thứ hai",
        2: "Thứ ba",
        3: "Thứ tư",
        4: "Thứ năm",
        5: "Thứ sáu",
        6: "Thứ bảy",
    }

    day_order = [
        "Chủ nhật",
        "Thứ hai",
        "Thứ ba",
        "Thứ tư",
        "Thứ năm",
        "Thứ sáu",
        "Thứ bảy",
    ]

    def __init__(self, cfg=CFG):
        self.cfg = cfg

    def valid_basket_size(self, prior_items):
        size = prior_items.groupby("order_id")["product_id"].nunique()
        return size[size >= self.cfg.min_basket_size]

    def basket_stats(self, prior_items):
        size = self.valid_basket_size(prior_items)
        labels = [
            "Nhỏ nhất",
            "Q1",
            "Trung vị",
            "Trung bình",
            "Q3",
            "Lớn nhất",
            "Độ lệch chuẩn",
        ]
        values = [
            int(size.min()),
            round(float(size.quantile(0.25)), 2),
            round(float(size.median()), 2),
            round(float(size.mean()), 2),
            round(float(size.quantile(0.75)), 2),
            int(size.max()),
            round(float(size.std()), 2),
        ]
        return pd.DataFrame({
            "Chỉ tiêu": labels,
            "Số sản phẩm": values,
        })

    def basket_ranges(self, prior_items):
        size = self.valid_basket_size(prior_items)
        bins = [1, 5, 10, 20, 40, np.inf]
        labels = ["2–5", "6–10", "11–20", "21–40", "Trên 40"]
        grouped = pd.cut(size, bins=bins, labels=labels, right=True)
        table = grouped.value_counts().reindex(labels).fillna(0).astype(int)
        result = table.reset_index()
        result.columns = ["Nhóm giỏ hàng", "Số đơn hàng"]
        result["Tỷ trọng (%)"] = (
            result["Số đơn hàng"] / len(size) * 100
        ).round(2)
        return result

    def orders_by_day(self, order_time, prior_items):
        orders = order_time[
            order_time["order_id"].isin(prior_items["order_id"])
        ]
        result = orders.groupby("order_dow")["order_id"].nunique().reset_index()
        result["Ngày trong tuần"] = result["order_dow"].map(self.day_names)
        result = result.rename(columns={"order_id": "Số đơn hàng"})
        result["Ngày trong tuần"] = pd.Categorical(
            result["Ngày trong tuần"],
            categories=self.day_order,
            ordered=True,
        )
        columns = ["Ngày trong tuần", "Số đơn hàng"]
        return result.sort_values("Ngày trong tuần")[columns]

    def orders_by_hour(self, order_time, prior_items):
        orders = order_time[
            order_time["order_id"].isin(prior_items["order_id"])
        ]
        result = orders.groupby("order_hour_of_day")["order_id"].nunique()
        result = result.reindex(range(24), fill_value=0).reset_index()
        return result.rename(columns={
            "order_hour_of_day": "Giờ trong ngày",
            "order_id": "Số đơn hàng",
        })

    def top_products(self, prior_items, catalog):
        columns = ["product_id", "Sản phẩm"]
        data = prior_items.merge(catalog[columns], on="product_id")
        result = data.groupby("Sản phẩm").size().reset_index(name="Số lần mua")
        return result.sort_values("Số lần mua", ascending=False).head(
            self.cfg.top_n
        )

    def top_categories(self, prior_items, catalog):
        columns = ["product_id", "Nhóm hàng lớn"]
        data = prior_items.merge(catalog[columns], on="product_id")
        result = data.groupby("Nhóm hàng lớn").size().reset_index(
            name="Số lần mua"
        )
        total = result["Số lần mua"].sum()
        result["Tỷ trọng trong tổng số lần mua (%)"] = (
            result["Số lần mua"] / total * 100
        ).round(2)
        return result.sort_values("Số lần mua", ascending=False).head(
            self.cfg.top_n
        )

    def reordered_products(self, prior_items, catalog):
        columns = ["product_id", "Sản phẩm"]
        data = prior_items.merge(catalog[columns], on="product_id")
        result = data.groupby("Sản phẩm").agg(
            **{
                "Số lần mua": ("order_id", "count"),
                "Số lần mua lại": ("reordered", "sum"),
            }
        ).reset_index()
        result["Tỷ lệ mua lại (%)"] = (
            result["Số lần mua lại"] / result["Số lần mua"] * 100
        ).round(2)
        return result.sort_values("Số lần mua lại", ascending=False).head(
            self.cfg.top_n
        )


## Khai thác luật

In [ ]:
class RuleMiner:
    def __init__(self, cfg=CFG):
        self.cfg = cfg
        self.base_transaction_count = 0
        self.rule_ready_transaction_count = 0
        self.min_support_count = 0
        self.kept_item_count = 0

    def transactions(self, prior_items):
        basket_size = prior_items.groupby("order_id")["product_id"].nunique()
        valid_orders = basket_size[basket_size >= self.cfg.min_basket_size].index
        data = prior_items[prior_items["order_id"].isin(valid_orders)].copy()

        self.base_transaction_count = len(valid_orders)
        self.min_support_count = ceil(
            self.cfg.min_support * self.base_transaction_count
        )

        item_count = data.groupby("product_id")["order_id"].nunique()
        keep_items = item_count[item_count >= self.min_support_count].index
        self.kept_item_count = len(keep_items)

        data = data[data["product_id"].isin(keep_items)].copy()
        data["product_id"] = data["product_id"].astype(str)

        baskets = data.groupby("order_id")["product_id"].apply(
            lambda items: tuple(sorted(set(items)))
        )
        transactions = [
            basket
            for basket in baskets
            if len(basket) >= self.cfg.min_basket_size
        ]

        self.rule_ready_transaction_count = len(transactions)
        return transactions

    def transaction_summary(self, prior_items, transactions):
        basket_size = prior_items.groupby("order_id")["product_id"].nunique()
        valid_orders = basket_size[basket_size >= self.cfg.min_basket_size]

        labels = [
            "Giỏ prior đủ điều kiện ban đầu",
            "Sản phẩm đủ phổ biến",
            "Giỏ còn đủ điều kiện sau lọc",
            "Số giỏ tối thiểu để đạt support",
        ]
        values = [
            int(len(valid_orders)),
            int(self.kept_item_count),
            int(len(transactions)),
            int(self.min_support_count),
        ]
        return pd.DataFrame({
            "Chỉ tiêu": labels,
            "Số lượng": values,
        })

    def support_base(self):
        return self.base_transaction_count or 1

    def pyfim_support_percent(self):
        total = max(self.rule_ready_transaction_count, 1)
        return self.min_support_count / total * 100

    def pyfim_itemsets(self, transactions, target="s"):
        output = fpgrowth(
            transactions,
            target=target,
            supp=self.pyfim_support_percent(),
            zmin=1,
            report="a",
        )
        return self.pyfim_to_itemsets(output)

    def pyfim_to_itemsets(self, output):
        rows = []
        total = self.support_base()

        for record in output:
            if len(record) < 2:
                continue

            itemset, count = record[0], record[1]
            if not isinstance(itemset, (list, tuple, set, frozenset)):
                itemset = (itemset,)

            rows.append([
                frozenset(map(str, itemset)),
                int(count) / total,
                int(count),
            ])

        columns = ["itemsets", "support", "support_count"]
        result = pd.DataFrame(rows, columns=columns)
        if result.empty:
            return result
        return result.sort_values("support", ascending=False).reset_index(
            drop=True
        )

    def rules_from_itemsets(self, itemsets):
        columns = [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift",
        ]
        if itemsets.empty:
            return pd.DataFrame(columns=columns)

        support_map = {
            frozenset(row["itemsets"]): row["support"]
            for _, row in itemsets.iterrows()
        }
        rows = []

        for itemset, support in support_map.items():
            if len(itemset) < 2:
                continue

            for size in range(1, len(itemset)):
                for left in combinations(itemset, size):
                    antecedents = frozenset(left)
                    consequents = itemset - antecedents
                    left_support = support_map.get(antecedents, 0)
                    right_support = support_map.get(consequents, 0)

                    if left_support == 0 or right_support == 0:
                        continue

                    confidence = support / left_support
                    lift = confidence / right_support

                    if confidence < self.cfg.min_confidence:
                        continue
                    if lift < self.cfg.min_lift:
                        continue

                    rows.append([
                        antecedents,
                        consequents,
                        support,
                        confidence,
                        lift,
                    ])

        result = pd.DataFrame(rows, columns=columns)
        if result.empty:
            return result
        return result.sort_values(
            ["lift", "confidence", "support"],
            ascending=False,
        ).reset_index(drop=True)

    def mine_fpgrowth(self, transactions):
        itemsets = self.pyfim_itemsets(transactions, target="s")
        rules = self.rules_from_itemsets(itemsets)
        return itemsets, rules

    def mine_apriori(self, transactions):
        output = apriori(
            transactions,
            target="s",
            supp=self.pyfim_support_percent(),
            zmin=1,
            report="a",
        )
        itemsets = self.pyfim_to_itemsets(output)
        rules = self.rules_from_itemsets(itemsets)
        return itemsets, rules

    def mine_eclat(self, transactions):
        output = eclat(
            transactions,
            target="s",
            supp=self.pyfim_support_percent(),
            zmin=1,
            report="a",
        )
        itemsets = self.pyfim_to_itemsets(output)
        rules = self.rules_from_itemsets(itemsets)
        return itemsets, rules

    def closed_and_maximal(self, transactions, frequent_itemsets):
        closed_itemsets = self.pyfim_itemsets(transactions, target="c")
        maximal_itemsets = self.pyfim_itemsets(transactions, target="m")

        return pd.DataFrame({
            "Loại itemset": [
                "Frequent itemsets",
                "Closed itemsets",
                "Maximal itemsets",
            ],
            "Số lượng": [
                int(len(frequent_itemsets)),
                int(len(closed_itemsets)),
                int(len(maximal_itemsets)),
            ],
        })

    def compare_algorithms(self, transactions, fp_result):
        rows = []
        fp_itemsets, fp_rules, fp_time = fp_result
        rows.append([
            "FP-Growth",
            len(fp_itemsets),
            len(fp_rules),
            round(fp_time, 2),
        ])

        for name, method in [
            ("Apriori", self.mine_apriori),
            ("ECLAT", self.mine_eclat),
        ]:
            start = perf_counter()
            itemsets, rules = method(transactions)
            rows.append([
                name,
                len(itemsets),
                len(rules),
                round(perf_counter() - start, 2),
            ])

        columns = [
            "Thuật toán",
            "Số frequent itemsets",
            "Số luật mua kèm",
            "Thời gian chạy (giây)",
        ]
        return pd.DataFrame(rows, columns=columns)

    def rules_for_storage(self, rules):
        result = rules.copy()
        result["antecedents"] = result["antecedents"].apply(
            lambda items: "|".join(sorted(items))
        )
        result["consequents"] = result["consequents"].apply(
            lambda items: "|".join(sorted(items))
        )
        return result

    def load_rules(self, path):
        data = load_csv(path)
        data["antecedents"] = data["antecedents"].apply(
            lambda value: frozenset(str(value).split("|"))
        )
        data["consequents"] = data["consequents"].apply(
            lambda value: frozenset(str(value).split("|"))
        )
        return data

## Định dạng kết quả

In [ ]:
class RuleFormatter:
    def __init__(self, catalog, cfg=CFG):
        self.cfg = cfg
        catalog = catalog.copy()
        catalog["product_id"] = catalog["product_id"].astype(str)
        self.name_map = dict(zip(catalog["product_id"], catalog["Sản phẩm"]))

    def item_name(self, item):
        return self.name_map.get(str(item), str(item))

    def itemset_text(self, itemset):
        names = [self.item_name(item) for item in sorted(map(str, itemset))]
        return ", ".join(names)

    def rule_text(self, row):
        left = self.itemset_text(row["antecedents"])
        right = self.itemset_text(row["consequents"])
        return f"{left} → {right}"

    def itemsets_table(self, itemsets):
        if itemsets.empty:
            return pd.DataFrame(columns=["Nhóm sản phẩm", "support"])

        result = itemsets[itemsets["itemsets"].apply(len) >= 2].copy()
        result = result.sort_values("support", ascending=False).head(
            self.cfg.top_n
        )
        result["Nhóm sản phẩm"] = result["itemsets"].apply(self.itemset_text)
        result["support"] = result["support"].round(4)
        return result[["Nhóm sản phẩm", "support"]]

    def rules_table(self, rules):
        columns = ["Luật", "support", "confidence", "lift"]
        if rules.empty:
            return pd.DataFrame(columns=columns)

        result = rules.sort_values(
            ["lift", "confidence", "support"],
            ascending=False,
        ).head(self.cfg.top_n)
        result = result.copy()
        result["Luật"] = result.apply(self.rule_text, axis=1)

        for column in ["support", "confidence", "lift", "weighted_score"]:
            if column in result.columns:
                result[column] = result[column].round(4)

        if "weighted_score" in result.columns:
            columns.append("weighted_score")

        return result[columns]

## Cải tiến luật

In [ ]:
class RuleImprover:
    def __init__(self, cfg=CFG):
        self.cfg = cfg

    def product_reorder_rate(self, prior_items):
        data = prior_items.copy()
        data["product_id"] = data["product_id"].astype(str)
        result = data.groupby("product_id").agg(
            times_bought=("order_id", "count"),
            times_reordered=("reordered", "sum"),
        )
        result["reorder_rate"] = result["times_reordered"] / result[
            "times_bought"
        ]
        return result["reorder_rate"].to_dict()

    def add_weighted_score(self, rules, prior_items):
        if rules.empty:
            return rules.copy()

        result = rules.copy()
        reorder_rate = self.product_reorder_rate(prior_items)

        result["reorder_score"] = result["consequents"].apply(
            lambda items: np.mean([
                reorder_rate.get(str(item), 0) for item in items
            ])
        )
        support_max = result["support"].max() or 1
        lift_max = result["lift"].max() or 1
        result["support_norm"] = result["support"] / support_max
        result["lift_norm"] = result["lift"] / lift_max
        result["weighted_score"] = (
            0.30 * result["support_norm"]
            + 0.30 * result["confidence"]
            + 0.30 * result["lift_norm"]
            + 0.10 * result["reorder_score"]
        )
        return result.sort_values("weighted_score", ascending=False)

    def valid_order_ids(self, items):
        basket_size = items.groupby("order_id")["product_id"].nunique()
        valid_orders = basket_size[basket_size >= self.cfg.min_basket_size]
        return set(valid_orders.index)

    def order_sets(self, items):
        data = items.copy()
        valid_orders = self.valid_order_ids(data)
        data = data[data["order_id"].isin(valid_orders)].copy()
        data["product_id"] = data["product_id"].astype(str)
        return data.groupby("order_id")["product_id"].apply(lambda x: set(x))

    def item_order_index(self, items):
        data = items.copy()
        valid_orders = self.valid_order_ids(data)
        data = data[data["order_id"].isin(valid_orders)].copy()
        data["product_id"] = data["product_id"].astype(str)
        grouped = data.groupby("product_id")["order_id"].apply(set)
        return grouped.to_dict(), valid_orders

    def orders_containing(self, item_index, itemset):
        order_sets = [item_index.get(str(item), set()) for item in itemset]

        if not order_sets or any(len(order_set) == 0 for order_set in order_sets):
            return set()

        result = set(order_sets[0])
        for order_set in order_sets[1:]:
            result &= order_set
        return result

    def hour_counts(self, order_ids, order_hour):
        if not order_ids:
            return {}

        hours = order_hour.reindex(list(order_ids)).dropna().astype(int)
        return hours.value_counts().to_dict()

    def metric_for_rule(self, order_sets, antecedents, consequents):
        antecedents = set(map(str, antecedents))
        consequents = set(map(str, consequents))
        left_count = 0
        right_count = 0
        both_count = 0
        total = len(order_sets)

        for items in order_sets:
            has_left = antecedents.issubset(items)
            has_right = consequents.issubset(items)

            if has_left:
                left_count += 1
            if has_right:
                right_count += 1
            if has_left and has_right:
                both_count += 1

        support = both_count / total if total else 0
        confidence = both_count / left_count if left_count else 0
        consequent_support = right_count / total if total else 0
        lift = confidence / consequent_support if consequent_support else 0
        return support, confidence, lift

    def validate_on_train(self, rules, prior_items, train_items):
        prior_sets = self.order_sets(prior_items)
        train_sets = self.order_sets(train_items)
        rows = []

        for _, row in rules.iterrows():
            prior_support, prior_confidence, prior_lift = self.metric_for_rule(
                prior_sets,
                row["antecedents"],
                row["consequents"],
            )
            train_support, train_confidence, train_lift = self.metric_for_rule(
                train_sets,
                row["antecedents"],
                row["consequents"],
            )
            rows.append([
                row["antecedents"],
                row["consequents"],
                round(prior_support, 4),
                round(train_support, 4),
                round(abs(prior_support - train_support), 4),
                round(prior_confidence, 4),
                round(train_confidence, 4),
                round(abs(prior_confidence - train_confidence), 4),
                round(prior_lift, 4),
                round(train_lift, 4),
                round(abs(prior_lift - train_lift), 4),
            ])

        columns = [
            "antecedents",
            "consequents",
            "prior_support",
            "train_support",
            "support_gap",
            "prior_confidence",
            "train_confidence",
            "confidence_gap",
            "prior_lift",
            "train_lift",
            "lift_gap",
        ]
        return pd.DataFrame(rows, columns=columns)


    def time_aware_rules(self, rules, prior_items, order_time):
        columns = [
            "antecedents",
            "consequents",
            "hour",
            "hour_support",
            "hour_confidence",
            "hour_lift",
            "antecedent_count",
            "rule_count",
        ]
        if rules.empty:
            return pd.DataFrame(columns=columns)

        item_index, valid_orders = self.item_order_index(prior_items)
        order_hour = order_time.set_index("order_id")["order_hour_of_day"]
        order_hour = order_hour[order_hour.index.isin(valid_orders)]
        hour_totals = order_hour.dropna().astype(int).value_counts().to_dict()
        rows = []

        for _, row in rules.iterrows():
            left_orders = self.orders_containing(
                item_index,
                row["antecedents"],
            )
            right_orders = self.orders_containing(
                item_index,
                row["consequents"],
            )
            both_orders = left_orders & right_orders

            if not left_orders or not right_orders or not both_orders:
                continue

            left_counts = self.hour_counts(left_orders, order_hour)
            right_counts = self.hour_counts(right_orders, order_hour)
            both_counts = self.hour_counts(both_orders, order_hour)

            for hour, both_count in both_counts.items():
                total = hour_totals.get(hour, 0)
                left_count = left_counts.get(hour, 0)
                right_count = right_counts.get(hour, 0)

                if not total or not left_count or not right_count:
                    continue

                hour_support = both_count / total
                hour_confidence = both_count / left_count
                consequent_support = right_count / total
                hour_lift = hour_confidence / consequent_support
                rows.append([
                    row["antecedents"],
                    row["consequents"],
                    int(hour),
                    round(hour_support, 4),
                    round(hour_confidence, 4),
                    round(hour_lift, 4),
                    int(left_count),
                    int(both_count),
                ])

        result = pd.DataFrame(rows, columns=columns)
        if result.empty:
            return result
        return result.sort_values(
            ["hour", "hour_confidence", "hour_lift"],
            ascending=[True, False, False],
        ).reset_index(drop=True)

## Vẽ biểu đồ

In [ ]:
class Plotter:
    def __init__(self, paths=PATHS):
        self.paths = paths

    def short_label(self, text, max_len=28):
        text = str(text)
        if len(text) <= max_len:
            return text
        return text[:max_len - 3] + "..."

    def label_value(self, value):
        if pd.isna(value):
            return ""
        if abs(value) >= 1000:
            return f"{int(round(value)):,}"
        return f"{value:.2f}"

    def max_min_indices(self, values):
        values = values.reset_index(drop=True)
        if values.empty:
            return []
        return sorted({int(values.idxmax()), int(values.idxmin())})

    def finish(self, fig, filename):
        fig.tight_layout()
        fig.savefig(self.paths.figure_file(filename), bbox_inches="tight")
        plt.show()
        plt.close(fig)

    def bar_vertical(self, data, x, y, title, filename):
        plot_data = data.copy().reset_index(drop=True)
        fig, ax = plt.subplots(figsize=(10, 5))

        if plot_data.empty:
            ax.text(0.5, 0.5, "Không có dữ liệu", ha="center", va="center")
            ax.set_title(title)
            self.finish(fig, filename)
            return

        bars = ax.bar(plot_data[x].astype(str), plot_data[y], label=y)
        ax.set_title(title)
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        ax.grid(axis="y", linestyle="--", alpha=0.35)
        ax.ticklabel_format(style="plain", axis="y")
        ax.legend()

        top = plot_data[y].max()
        ax.set_ylim(0, top * 1.15 if top else 1)

        for index in self.max_min_indices(plot_data[y]):
            bar = bars[index]
            value = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value,
                self.label_value(value),
                ha="center",
                va="bottom",
                fontsize=9,
            )

        self.finish(fig, filename)

    def bar_horizontal(self, data, x, y, title, filename):
        plot_data = data.copy().reset_index(drop=True)
        fig, ax = plt.subplots(figsize=(10, 6))

        if plot_data.empty:
            ax.text(0.5, 0.5, "Không có dữ liệu", ha="center", va="center")
            ax.set_title(title)
            self.finish(fig, filename)
            return

        plot_data[x] = plot_data[x].apply(self.short_label)
        bars = ax.barh(plot_data[x], plot_data[y], label=y)
        ax.set_title(title)
        ax.set_xlabel(y)
        ax.set_ylabel(x)
        ax.grid(axis="x", linestyle="--", alpha=0.35)
        ax.ticklabel_format(style="plain", axis="x")
        ax.legend()
        ax.invert_yaxis()

        top = plot_data[y].max()
        ax.set_xlim(0, top * 1.30 if top else 1)

        for index in self.max_min_indices(plot_data[y]):
            bar = bars[index]
            value = bar.get_width()
            ax.text(
                value + top * 0.04,
                bar.get_y() + bar.get_height() / 2,
                self.label_value(value),
                ha="left",
                va="center",
                fontsize=9,
                clip_on=False,
            )

        self.finish(fig, filename)

    def histogram(self, data, column, title, filename, threshold=None):
        values = data[column].dropna()
        fig, ax = plt.subplots(figsize=(9, 5))

        if values.empty:
            ax.text(0.5, 0.5, "Không có dữ liệu", ha="center", va="center")
        else:
            ax.hist(
                values,
                bins=15,
                edgecolor="black",
                linewidth=0.8,
                alpha=0.85,
                label=column,
            )

        if threshold is not None:
            ax.axvline(
                threshold,
                linestyle="--",
                linewidth=2.2,
                label=f"Ngưỡng: {threshold}",
            )

        ax.set_title(title)
        ax.set_xlabel(column)
        ax.set_ylabel("Số luật")
        ax.grid(axis="y", linestyle="--", alpha=0.35)
        ax.ticklabel_format(style="plain", axis="y")
        ax.legend()
        self.finish(fig, filename)

    def scatter_reference(self, data, x, y, title, filename):
        fig, ax = plt.subplots(figsize=(7, 6))

        if data.empty:
            ax.text(0.5, 0.5, "Không có dữ liệu", ha="center", va="center")
        else:
            ax.scatter(data[x], data[y], label="Luật mua kèm")
            low = min(data[x].min(), data[y].min())
            high = max(data[x].max(), data[y].max())
            ax.plot(
                [low, high],
                [low, high],
                linestyle="--",
                label="Prior = Train",
            )

        ax.set_title(title)
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.legend()
        self.finish(fig, filename)


PLOT = Plotter()